In [1]:
import xarray as xr
import numpy as np
import dask
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [2]:
cluster = PBSCluster(
    cores=1,
    memory='64GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='1:00:00'
)
cluster.scale(jobs=4)
client = Client(cluster)
client

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45149 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/45149/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/45149/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.97:37707,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/45149/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
path='/glade/work/acruz/Caribbean_Heat_data/ERA5/'

In [4]:
MVIMD = xr.open_zarr(path+'MVIMD')
MVIMD

<xarray.Dataset> Size: 16GB
Dimensions:                (forecast_initial_time: 33052, forecast_hour: 12,
                            latitude: 82, longitude: 121)
Coordinates:
  * forecast_initial_time  (forecast_initial_time) datetime64[ns] 264kB 1981-...
  * forecast_hour          (forecast_hour) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
Data variables:
    MVIMD                  (forecast_initial_time, forecast_hour, latitude, longitude) float32 16GB dask.array<chunksize=(8263, 12, 82, 121), meta=np.ndarray>

In [5]:
MVIMD_stack = MVIMD.stack(time=('forecast_initial_time', 'forecast_hour'))
actual_time = MVIMD_stack['forecast_initial_time'] + MVIMD_stack['forecast_hour'].astype('timedelta64[h]')

MVIMD_fxtime = MVIMD_stack.assign_coords(fixed_time=actual_time).sortby('time').drop_duplicates('time')
MVIMD_fxtime

<xarray.Dataset> Size: 16GB
Dimensions:                (latitude: 82, longitude: 121, time: 396624)
Coordinates:
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
  * time                   (time) object 3MB MultiIndex
  * forecast_initial_time  (time) datetime64[ns] 3MB 1981-01-01T06:00:00 ... ...
  * forecast_hour          (time) int32 2MB 1 2 3 4 5 6 7 8 ... 6 7 8 9 10 11 12
    fixed_time             (time) datetime64[ns] 3MB 1981-01-01T07:00:00 ... ...
Data variables:
    MVIMD                  (latitude, longitude, time) float32 16GB dask.array<chunksize=(82, 121, 99156), meta=np.ndarray>

In [6]:
MVIMD_clim = MVIMD_fxtime.groupby('fixed_time.month').mean()
MVIMD_clim

<xarray.Dataset> Size: 478kB
Dimensions:    (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    MVIMD      (month, latitude, longitude) float32 476kB dask.array<chunksize=(12, 82, 121), meta=np.ndarray>

In [ ]:
MVIMD_clim.to_netcdf(path+'MVIMD_clim.nc')

In [ ]:
clien.shutdown()